# FightWatch — train lớn trên Drive (RLVS + AIRTLab)

Chỉ phát hiện đánh nhau. Không dùng NTU hoặc frame GT UBI đang bị cảnh báo. Tách nhóm **train / development / calibration / test** trước khi tạo cửa sổ. Ngưỡng chọn bằng calibration, không tối ưu trên test. Điểm clip không phải độ chính xác sự kiện/camera iPhone.

RLVS: [nguồn Kaggle](https://www.kaggle.com/datasets/mohamedmustafa/real-life-violence-situations-dataset). AIRTLab: [nguồn và trích dẫn](https://github.com/airtlab/A-Dataset-for-Automatic-Violence-Detection-in-Videos), Bianculli et al. 2020, DOI 10.1016/j.dib.2020.106587; dùng nghiên cứu/giáo dục. Bỏ clip AIRTLab chứa club/stab/gunshot, giữ âm khó ôm/vỗ tay/vung tay. Hai camera cùng màn diễn luôn chung partition. AIRTLab vẫn chung phòng/diễn viên; không tuyên bố actor-independent.

Ngân sách training/cache 45 phút, watchdog thêm 60 giây để dừng tiến trình; tải dữ liệu là bước riêng. Không tự chạy lại vòng dài nếu thất bại.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
ROOT = Path('/content/drive/MyDrive/SENTINEL')
REPO = ROOT / 'repo/anomaly-behavior-detector-system'
LOCAL = Path('/content/fightwatch-large-code')
ARCHIVES = ROOT / 'datasets/fight'
DATA = Path('/content/fightwatch-public-data')
OUTPUT = ROOT / 'artifacts/fight_large_v1'
CACHE = Path('/content/fightwatch-large-cache')
BRANCH = 'main'
print('Run:', OUTPUT)

## Cập nhật code từ GitHub
Chỉ bỏ thay đổi tự lưu của notebook này trước khi pull. Nếu file code khác trên Drive bị sửa, cell dừng để bảo vệ dữ liệu. Sau pull, code chạy trong tiến trình Python mới nên không giữ module cũ trong kernel.

In [ ]:
import subprocess, shutil, sys, os
if not (REPO / '.git').exists():
    REPO.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git','clone','--branch',BRANCH,'https://github.com/TranDuon/anomaly-behavior-detector-system.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'restore','--','colab/fight_large_drive.ipynb'],check=True)
changed = subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True)
if changed.strip():
    raise RuntimeError('File khác có thay đổi; không tự ghi đè:\n'+changed)
subprocess.run(['git','-C',str(REPO),'pull','--ff-only','origin',BRANCH],check=True)
REVISION = subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
shutil.copytree(REPO,LOCAL,dirs_exist_ok=True,ignore=shutil.ignore_patterns('.git','.venv','logs','uploads','models','__pycache__','.pytest_cache'))
os.chdir(LOCAL)
print('Training revision:', REVISION)

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-colab.txt'],check=True)
import torch
assert torch.cuda.is_available(), 'Chọn GPU trong Runtime > Change runtime type'
print(torch.cuda.get_device_name(0))
from scripts.colab_process import run_logged

## Tải archive một lần, kiểm CRC rồi giải nén vào /content
Không coi file đang tải dở là dataset hoàn chỉnh. RLVS archive có hai bản thư mục lặp: adapter chỉ dùng một bộ 2.000 video. File archive và manifest lưu Drive; video giải nén/cache RGB ở /content có thể tạo lại khi mất runtime.

In [ ]:
run_logged([sys.executable,'-m','scripts.fight_public_data','--archives',str(ARCHIVES),'--data',str(DATA),'--output',str(OUTPUT)],OUTPUT/'data.log',3600,revision=REVISION)
import json
manifest = json.loads((OUTPUT/'manifest.json').read_text())
print(json.dumps(manifest['counts'],indent=2))
print(manifest['label_provenance'])

## Kiểm tra nhãn trước khi train
Các trang dưới là mẫu ngẫu nhiên train/development, ẩn tên nguồn/nhãn/score. Không xem test để quyết định mô hình. Đây chỉ là audit mẫu, không biến toàn bộ nhãn clip thành nhãn hành vi thời gian đã được người kiểm duyệt. Nếu nhiều clip lệch tiêu chí, dừng và tạo phiên bản manifest mới trước training.

In [ ]:
from IPython.display import display, Image
for page in sorted((OUTPUT/'audit').glob('page*.jpg')):
    display(Image(filename=str(page)))

## Train + chọn checkpoint + hiệu chỉnh + test một lần
X3D-S pretrained Kinetics400, ROI theo cụm người/tube + toàn cảnh, fine-tune có rung/biến đổi nhẹ. Không bắt buộc detector thấy đủ hai người (che khuất/cận cảnh có thể mất detection). Mục tiêu precision và recall cùng ≥80%, không chỉ accuracy. Nếu không đạt, báo thất bại, không tự gắn `approved_for_demo`.

Không chạy lại vào cùng OUTPUT đã có run/report. Thay hyperparameter sau khi xem test sẽ làm test trở thành dữ liệu phát triển; cần holdout mới để xác nhận.

In [ ]:
run_logged([sys.executable,'-m','scripts.fight_large_train','--manifest',str(OUTPUT/'manifest.json'),'--output',str(OUTPUT),'--cache',str(CACHE),'--budget-minutes','45'],OUTPUT/'training.log',2760,revision=REVISION)

In [ ]:
report = json.loads((OUTPUT/'report.json').read_text())
print(json.dumps({key:report[key] for key in ['test','test_by_dataset','public_clip_target_met','decision','real_phone_tested','seconds']},indent=2))
print('CPU bundle:',OUTPUT/'sentinel_fight_only_runtime.zip')

## Mang về local
Gói chứa model TorchScript và metadata/ngưỡng đúng phiên bản. Chạy `run_local.cmd` ở repo local để cài bundle. Model vẫn là thử nghiệm: bật chế độ pilot có chủ đích. Thử video offline trước, sau đó chọn **Phát lại theo thời gian thực** để đo bộ đệm/độ trễ CPU. Chỉ xác nhận camera iPhone sau khi điện thoại đã nối, không suy ra từ clip-test.

Checkpoint, manifest, lịch sử, dự đoán test và log lưu Drive; frozen_features.zip chỉ lưu đặc trưng nén, cache RGB tạm không được coi là bản sao lưu lâu dài. Ngắt runtime sau khi đã kiểm tra đủ file (không tự xóa runtime đang xem).

In [ ]:
from google.colab import files
bundle = OUTPUT/'sentinel_fight_only_runtime.zip'
assert bundle.exists()
files.download(str(bundle))